# 表达式与计算性能

学习目标：为表格计算选择合适的内置操作、函数应用或迭代，安全使用固定表达式，先核对结果再测量性能，识别引擎、类型和准备成本的限制。

前置知识：函数、向量化、性能测量、结果比较。

运行环境：Python 3.12、pandas 3、NumPy、NumExpr；Numba 与 Cython 仅介绍用途，不安装或运行。

环境准备：[环境配置与运行](README.md)

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章为扩展内容，计时仅使用 20 行和 20000 行的自制数据，耗时随机器及当时负载变化。

## 1 先用内置运算完成任务

假设为三条测量记录计算一个自制指标：score = x² + 2y。其中 x、y 是无单位的输入数值，score 也没有物理单位。这里用整列算术即可表达，不需要逐行调用 Python 函数。

将计算封装为一个短函数，只为后面复用同一任务进行比较；本节立即用小表检查结果。索引故意采用 B、A、C 的顺序，便于发现计算是否丢失或重排标签。

In [1]:
import warnings
from timeit import repeat

import numexpr as ne
import numpy as np
import pandas as pd

measurements = pd.DataFrame(
    {"x": [1.0, 2.0, 3.0], "y": [2.0, 1.0, 4.0]}, index=["B", "A", "C"],
)


def score_builtin(frame):
    return (frame["x"] ** 2 + 2 * frame["y"]).rename("score")


expected = score_builtin(measurements)
print(expected)
# B、A、C 分别得到 5、6、17；顺序保持不变，结果为 float64。

B     5.0
A     6.0
C    17.0
Name: score, dtype: float64


## 2 map、apply 与逐行迭代

Series.map 按元素映射，参数可以是函数、字典或 Series，适合逐值替换；apply(axis=1) 默认把每一行作为 Series 传给函数，适合必须结合多列的自定义逻辑。能用现有整列操作表达的计算，优先保留整列写法。

下面为了比较，同一个简单公式分别使用逐值函数和逐行函数。默认的函数调用不会因为写成 map 或 apply 就自动编译成机器码。

In [2]:
def score_map(frame):
    squared = frame["x"].map(lambda value: value ** 2)
    return (squared + 2 * frame["y"]).rename("score")


def score_apply(frame):
    return frame.apply(
        lambda row: row["x"] ** 2 + 2 * row["y"], axis=1,
    ).rename("score")


for method in [score_map, score_apply]:
    result = method(measurements)
    pd.testing.assert_series_equal(result, expected)
    print(method.__name__, result.tolist(), result.dtype)
# 两种写法都得到相同值、标签和 dtype；是否值得使用还取决于任务与实测成本。

score_map [5.0, 6.0, 17.0] float64
score_apply [5.0, 6.0, 17.0] float64


确实要逐条调用 Python 逻辑时，可以用 itertuples 取得行元组。index=False 不把索引放入元组，name=None 返回普通元组；下面明确只取 x、y 两列，并在构造结果时放回原索引。

这个例子用于观察迭代的成本，不是建议把简单算术改写成循环。np.vectorize 也只是方便包装逐元素调用，其实现本质上是循环，不是把 Python 函数编译成快速内核。

In [3]:
def score_iterate(frame):
    values = [
        x ** 2 + 2 * y
        for x, y in frame[["x", "y"]].itertuples(index=False, name=None)
    ]
    return pd.Series(values, index=frame.index, name="score", dtype="float64")


iterated = score_iterate(measurements)
pd.testing.assert_series_equal(iterated, expected)
print(iterated)
# 原标签与顺序被显式放回，不能只生成一个没有行标签的列表就声称结果完全相同。

B     5.0
A     6.0
C    17.0
Name: score, dtype: float64


## 3 eval 表达列计算

DataFrame.eval 使用字符串描述列运算，列名可以直接写在表达式中。engine="python" 使用 Python 求值路径，engine="numexpr" 请求 NumExpr；后者针对其支持的数值表达式执行计算，不是通用 Python 解释器。

这里的表达式由作者固定写出。eval 和 query 都可能执行任意代码，不应接收用户提供的表达式文本；切换 engine 也不能把它们变成安全沙箱。

In [4]:
def score_eval_python(frame):
    return frame.eval("x ** 2 + 2 * y", engine="python").rename("score")


def score_eval_numexpr(frame):
    return frame.eval("x ** 2 + 2 * y", engine="numexpr").rename("score")


for method in [score_eval_python, score_eval_numexpr]:
    evaluated = method(measurements)
    pd.testing.assert_series_equal(evaluated, expected)
    print(method.__name__, evaluated.tolist(), evaluated.dtype)
# 两种引擎在本例都返回 float64 Series；小表中能运行不代表会更快。

score_eval_python [5.0, 6.0, 17.0] float64
score_eval_numexpr [5.0, 6.0, 17.0] float64


eval 也支持列赋值。表达式中的列名来自表，局部变量前面加 @；local_dict 可以明确提供这些变量。默认 inplace=False，带赋值的表达式返回新表，不直接修改原表。

In [5]:
weighted = measurements.eval(
    "score = x ** 2 + @weight * y",
    local_dict={"weight": 2.0}, engine="numexpr",
)
print(weighted)
print(measurements.columns.tolist())
# 新表增加 score，原表仍只有 x、y；@weight 指向给出的局部数值。

     x    y  score
B  1.0  2.0    5.0
A  2.0  1.0    6.0
C  3.0  4.0   17.0
['x', 'y']


## 4 query 表达筛选条件

query 使用布尔表达式筛选行，保留相应的索引与列。下面固定条件模板，只通过 local_dict 传入两个数值阈值；不把阈值拼成表达式文本。

query 默认采用 pandas 解析器，允许把整列条件用 and、or 组合；这是表达式字符串的语法。直接写 pandas 布尔掩码时，仍用带括号的 &、|。parser 决定语法规则，engine 决定执行方式，两者不同。

In [6]:
selected = measurements.query(
    "x >= @minimum and y < @maximum",
    local_dict={"minimum": 2.0, "maximum": 4.0}, engine="numexpr",
)
direct = measurements.loc[(measurements["x"] >= 2.0) & (measurements["y"] < 4.0)]
pd.testing.assert_frame_equal(selected, direct)
print(selected)
# 只留下 A 行；标签、两列、dtype 和值均与直接筛选一致。

     x    y
A  2.0  1.0


## 5 NumExpr 的类型边界

NumExpr 内部支持有限的类型，包括布尔、32/64 位有符号整数、32/64 位浮点和复数等。更小的整数会提升位宽，任意 Python 对象不是它的原生数组类型。

先观察两个 pandas 求值引擎：同样的 int8 输入和加法，数值相同，结果 dtype 却可以不同。所以只比较打印出的数字不足以检查等价性。

In [7]:
small_ints = pd.DataFrame({"x": np.array([1, 2], dtype="int8")})
python_result = small_ints.eval("x + x", engine="python")
numexpr_result = small_ints.eval("x + x", engine="numexpr")
print(python_result.tolist(), python_result.dtype)
print(numexpr_result.tolist(), numexpr_result.dtype)
# Python 路径为 int8，NumExpr 路径为 int32；数值都是 [2, 4]，但类型不相同。

[2, 4] int8
[2, 4] int32


下面直接调用 NumExpr，只用来明确观察 object 数组的限制。即使数组里目前装的都是整数，dtype=object 仍不等于原生整数数组。若要转换类型，应先确认数值范围、缺失语义和转换成本。

In [8]:
objects = np.array([1, 2], dtype=object)
try:
    ne.evaluate("x + 1", local_dict={"x": objects})
except ValueError:
    print("ValueError：NumExpr 不支持这个 object 数组")
else:
    raise AssertionError("应拒绝 object dtype")
native = objects.astype("int64")
print(ne.evaluate("x + 1", local_dict={"x": native}))
# 明确转换后得到 [2 3]；这并不表示任意混合对象都可这样转换。

ValueError：NumExpr 不支持这个 object 数组
[2 3]


## 6 识别引擎回退

pandas 3.0.6 对下面的可空 Int64 表达式，即使指定 engine="numexpr"，也会发出 RuntimeWarning 并切换到 Python 引擎。警告说明请求与实际执行路径不同，不能忽略后把结果记作 NumExpr 的性能。

只在这一小段记录警告，并检查类别与内容。随后显式使用 Python 引擎，核对缺失、标签和类型都未改变；其他非预期警告不在这里放过。

In [9]:
nullable = pd.DataFrame({"x": pd.array([1, None, 3], dtype="Int64")}, index=["A", "B", "C"])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fallback = nullable.eval("x + 1", engine="numexpr")
assert len(caught) == 1
assert caught[0].category is RuntimeWarning
assert "Engine has switched to 'python'" in str(caught[0].message)
print(caught[0].category.__name__, str(caught[0].message))
explicit_python = nullable.eval("x + 1", engine="python")
pd.testing.assert_series_equal(fallback, explicit_python)
print(explicit_python)
# 结果为 2、<NA>、4，dtype 仍为 Int64；本次运算实际使用 Python 路径。

RuntimeWarning Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
A       2
B    <NA>
C       4
Name: x, dtype: Int64


## 7 先检查，再确定计时口径

下面比较同一公式的六种实现，输入均为无缺失 float64。测试规模为 20 行和 20000 行；pandas 指南把超过约 10000 行作为考虑 eval 的经验起点，不是保证加速的阈值，表达式复杂度也影响收益。

每个实现先核对索引、名称、dtype 和数值，再进入计时。浮点数采用相对与绝对容差 1e-12：输入只在 0 至 3 附近，计算步骤少，此容差用于允许浮点舍入的微小差异，不允许以改 dtype 或漏行换速度。

In [10]:
methods = {
    "builtin": score_builtin,
    "map": score_map,
    "apply": score_apply,
    "itertuples": score_iterate,
    "eval_python": score_eval_python,
    "eval_numexpr": score_eval_numexpr,
}
bench_frames = {}
for size in [20, 20000]:
    data = pd.DataFrame({
        "x": np.linspace(0.0, 2.0, size),
        "y": np.linspace(1.0, 3.0, size),
    }, index=pd.RangeIndex(size - 1, -1, -1, name="row"))
    bench_frames[size] = data
    reference = score_builtin(data)
    for method in methods.values():
        pd.testing.assert_series_equal(
            method(data), reference, check_exact=False, rtol=1e-12, atol=1e-12,
        )
    print(size, "行：六种结果一致", reference.dtype)
# 反向行标签也参与检查；两种规模都通过后再计时。

20 行：六种结果一致 float64


20000

 行：六种结果一致 float64


计时从已经准备好的 DataFrame 开始，包含函数调用、表达式处理、计算与结果 Series 构造；不含导入、输入生成、dtype 转换、结果检查和打印。上面的正确性检查也完成了预热，因此不代表首次冷启动耗时；NumExpr 会缓存表达式，但每次 pandas 调用仍有处理开销。

使用 timeit.repeat 做 3 组测量，每组调用 3 次，取最快一组除以 3，换算成毫秒。它是当前条件下较少受干扰的每次耗时估计，不是生产请求延迟保证。timeit 默认暂时关闭垃圾回收，这个口径也不等于完整应用运行。

In [11]:
timings = []
print("NumExpr 当前线程数：", ne.get_num_threads())
for size, data in bench_frames.items():
    for name, method in methods.items():
        samples = repeat(lambda: method(data), repeat=3, number=3)
        timings.append({"rows": size, "method": name, "ms_per_call": min(samples) / 3 * 1000})
timing_table = pd.DataFrame(timings)
print(timing_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
# 记录实际耗时；不预设哪种实现胜出，也不把一个规模的排序推广到所有输入。

NumExpr 当前线程数： 16


 rows       method  ms_per_call
   20      builtin       0.2099
   20          map       0.2120
   20        apply       0.2764
   20   itertuples       0.5114
   20  eval_python       0.7880
   20 eval_numexpr       1.1946
20000      builtin       0.2359
20000          map       4.8120
20000        apply     119.3593
20000   itertuples       7.4561
20000  eval_python       0.8928
20000 eval_numexpr       1.5739


## 8 把准备成本纳入比较

上面的结果不包含将字符串解析成数值的成本。若实际输入是文本，就应把转换加入要比较的完整步骤。下面只比较同一个内置公式在“已有 float64 输入”和“每次先转换文本输入”两种口径下的耗时。

输入文本在计时外构造；第二种计时包含 astype 转换及计算，不含文件读取。仍然先比较结果，再测量。这样才能明确一个更短的核心计算是否被准备开销抵消。

In [12]:
prepared = bench_frames[20000]
text_input = prepared.astype("str")


def parse_and_compute():
    converted = text_input.astype("float64")
    return score_builtin(converted)


pd.testing.assert_series_equal(
    parse_and_compute(), score_builtin(prepared),
    check_exact=False, rtol=1e-12, atol=1e-12,
)
for label, task in [
    ("already_float64", lambda: score_builtin(prepared)),
    ("parse_then_compute", parse_and_compute),
]:
    elapsed = min(repeat(task, repeat=3, number=3)) / 3 * 1000
    print(label, f"{elapsed:.4f} ms/call")
# 两个结果相同；两项耗时的差别包含字符串到浮点数的准备过程。

already_float64 0.2399 ms/call
parse_then_compute 12.9634 ms/call


## 9 选学：Numba 与 Cython 的入口

Numba 可把支持的数值函数按参数类型编译为机器码，适合难以用现有数组操作表达、又被重复调用的数值循环。首次编译有成本，评估时须分别记录首次调用与复用已编译代码的耗时；支持的 Python、NumPy 操作也有限。

Cython 将代码转换为 C 等编译代码，可通过静态类型减少部分 Python 操作开销，也能连接 C 库。它需要编译工具与额外构建步骤，收益应来自已经定位的计算瓶颈。

两者都不是为短小计算自动加速的通用按钮。先确定输入类型、正确性标准与收益目标，再决定是否增加编译依赖。本章不导入或运行这两种工具；np.vectorize 与 Numba 的编译装饰器也不是同一种机制。

## 本章小结

（1）已有整列操作能完成的任务，优先采用内置写法；map、apply 和迭代分别服务于不同输入粒度，不应只按 API 名称判断速度。

（2）eval、query 只使用可信的固定表达式，外部数据通过变量提供；解析器和计算引擎有不同职责。

（3）检查 dtype 变化、失败和引擎回退。输出有正确数字，不代表标签、缺失语义或实际执行引擎都符合预期。

（4）先比较完整结果，再在明确规模和成本范围内计时；计时结果不等于固定加速倍数，准备与首次编译成本也应按真实任务考虑。

## 练习

（1）计算下面每行的 x + 2y，分别用内置操作与 apply 实现。先检查两者的标签、dtype 和值，不做计时。

In [13]:
practice = pd.DataFrame({"x": [2.0, 4.0], "y": [3.0, 1.0]}, index=["P2", "P1"])
# 补充：两种结果都命名为 score，再用 assert_series_equal 检查。
# 检查：P2、P1 的结果为 8、6，不能重排原索引。

（2）先预测两个 query 分别保留哪些行，再运行。说明同名列 limit 与局部变量 limit 为什么有不同含义。

In [14]:
prediction = pd.DataFrame({"x": [1, 3, 5], "limit": [0, 4, 6]}, index=["A", "B", "C"])
limit = 2
print(prediction.query("x > limit", engine="python").index.tolist())
print(prediction.query("x > @limit", engine="python").index.tolist())
# 补充：记录预测，并解释 @ 的作用；表达式均为作者固定内容。

['A']
['B', 'C']


（3）原输入没有缺失，后来改成可空 Int64，并要求保留缺失和整数类型。有人建议转成普通 float64 再强制 NumExpr。你会选择这种转换、显式 Python 引擎，还是内置运算？说明理由，给出符合要求的结果并核对 dtype。

In [15]:
changed = pd.DataFrame({"x": pd.array([2, None, 4], dtype="Int64")})
# 任务：计算 x + 1，结果须为 3、<NA>、5，且 dtype 仍为 Int64。
# 补充：给出选择理由；若观察回退警告，应局部捕获并验证，不全局忽略。

（4）有一份报告只计时 NumExpr 的核心计算，却让另一种实现承担文本解析。请指出比较口径的问题，使用下面的小输入为内置运算与 eval 设计公平的计时：两者都包含或都不包含转换，并说明预热与重复次数。

In [16]:
raw_for_timing = pd.DataFrame({"x": ["1.5", "2.5"] * 10, "y": ["2.0", "3.0"] * 10})
# 补充：选定统一口径；先核对 x ** 2 + 2 * y 的结果，再 repeat=3、number=3。
# 检查：不能用关闭 dtype 检查、丢失标签或漏掉准备步骤换取表面上的更短耗时。
# 说明：本题只测 20 行，不能据此断言在大表上也有相同排序。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | pandas 3.0.6 [Enhancing performance](https://pandas.pydata.org/docs/user_guide/enhancingperf.html)：Cython、Numba、Expression evaluation via eval、Local variables、parsers 与规模条件；[Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)、[DataFrame.apply](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)、[itertuples](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html) 的输入粒度、axis、index 与 name；[DataFrame.eval](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.eval.html)、[query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) 的 Warning 与 inplace；[pandas.eval](https://pandas.pydata.org/docs/reference/api/pandas.eval.html) 的 engine、parser、local_dict 参数（DataFrame.eval 通过 kwargs 转交）；[assert_series_equal](https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_series_equal.html) 的 dtype、索引、名称、rtol、atol 检查。扩展类型回退警告另核查本环境官方发行包 pandas 3.0.6 的 pandas/core/computation/eval.py：eval 中 extension array dtype 检查与 RuntimeWarning 分支。 |
| NumExpr 官方文档 | [User Guide](https://numexpr.readthedocs.io/en/latest/user_guide.html) 的 Usage Notes、Datatypes supported internally、Casting rules、Threadpool configuration：表达式缓存、类型集合与升位。页面标识为 2.13.dev1；实际环境为 2.14.2，另核查官方发行包 numexpr/necompiler.py 的 getType，并实际验证 int8 升位与 object 拒绝。 |
| NumPy 官方文档 | NumPy 2.5 [vectorize](https://numpy.org/doc/2.5/reference/generated/numpy.vectorize.html) 的 Notes：便利包装与循环实现，不等同于编译加速。 |
| Python 官方文档 | Python 3.12.14 [timeit](https://docs.python.org/3.12/library/timeit.html) 的 repeat、重复测量取最佳组、调用次数与垃圾回收条件；[warnings：Testing Warnings](https://docs.python.org/3.12/library/warnings.html#testing-warnings) 的局部记录、类别及内容检查。 |
| Numba 官方文档 | [A ~5 minute guide](https://numba.readthedocs.io/en/stable/user/5minguide.html) 的 nopython 编译、How to measure the performance of Numba、How does Numba work：参数类型、首次编译和缓存复用；仅用于选学入口。 |
| Cython 官方文档 | Cython 3.3.0 [Cython - an overview](https://docs.cython.org/en/latest/src/quickstart/overview.html)：编译、静态类型、扩展模块及与 C 库互操作；仅用于选学入口。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[enhancingperf](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/enhancingperf.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |